In [ ]:
import torch
import torch.nn as nn
import cv2
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
import torch.nn.functional as F
from IPython.display import display, clear_output
import ipywidgets as widgets

# ==========================================
# 1. SHARED ARCHITECTURE BLOCKS
# ==========================================
class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)
    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x

def window_partition(x, window_size):
    B, D, H, W, C = x.shape
    x = x.view(B, D // window_size[0], window_size[0], H // window_size[1], window_size[1], W // window_size[2], window_size[2], C)
    windows = x.permute(0, 1, 3, 5, 2, 4, 6, 7).contiguous().view(-1, window_size[0], window_size[1], window_size[2], C)
    return windows

def window_reverse(windows, window_size, D, H, W):
    B = int(windows.shape[0] / (D * H * W / window_size[0] / window_size[1] / window_size[2]))
    x = windows.view(B, D // window_size[0], H // window_size[1], W // window_size[2], window_size[0], window_size[1], window_size[2], -1)
    x = x.permute(0, 1, 4, 2, 5, 3, 6, 7).contiguous().view(B, D, H, W, -1)
    return x

class WindowAttention3D(nn.Module):
    def __init__(self, dim, window_size, num_heads):
        super().__init__()
        self.dim = dim
        self.window_size = window_size
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3, bias=True)
        self.proj = nn.Linear(dim, dim)
        self.softmax = nn.Softmax(dim=-1)
        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2 * window_size[0] - 1) * (2 * window_size[1] - 1) * (2 * window_size[2] - 1), num_heads)) 
    def forward(self, x):
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = self.softmax(attn)
        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        x = self.proj(x)
        return x

class SwinTransformerBlock3D(nn.Module):
    def __init__(self, dim, num_heads, window_size=(2,4,4), shift_size=(0,0,0)):
        super().__init__()
        self.dim = dim
        self.window_size = window_size
        self.shift_size = shift_size
        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowAttention3D(dim, window_size, num_heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = Mlp(in_features=dim, hidden_features=int(dim * 4.))
    def forward(self, x):
        B, D, H, W, C = x.shape
        shortcut = x
        x = self.norm1(x)
        if sum(self.shift_size) > 0:
            shifted_x = torch.roll(x, shifts=(-self.shift_size[0], -self.shift_size[1], -self.shift_size[2]), dims=(1, 2, 3))
        else: shifted_x = x
        x_windows = window_partition(shifted_x, self.window_size)
        x_windows = x_windows.view(-1, self.window_size[0] * self.window_size[1] * self.window_size[2], C)
        attn_windows = self.attn(x_windows)
        attn_windows = attn_windows.view(-1, self.window_size[0], self.window_size[1], self.window_size[2], C)
        shifted_x = window_reverse(attn_windows, self.window_size, D, H, W)
        if sum(self.shift_size) > 0:
            x = torch.roll(shifted_x, shifts=(self.shift_size[0], self.shift_size[1], self.shift_size[2]), dims=(1, 2, 3))
        else: x = shifted_x
        x = shortcut + x
        x = x + self.mlp(self.norm2(x))
        return x

# ==========================================
# 2. MODEL A: DENOISER
# ==========================================
class OCT3DDenoisingAutoencoder(nn.Module):
    def __init__(self, in_chans=1, embed_dim=96, patch_size=4, depths=[2, 2], num_heads=[3, 6]):
        super().__init__()
        self.num_layers = len(depths)
        self.embed_dim = embed_dim
        self.patch_embed = nn.Conv3d(in_chans, embed_dim, kernel_size=(1, patch_size, patch_size), stride=(1, patch_size, patch_size))
        self.layers = nn.ModuleList()
        for i_layer in range(self.num_layers):
            layer = nn.Sequential(
                SwinTransformerBlock3D(dim=embed_dim, num_heads=num_heads[i_layer], window_size=(2,4,4), shift_size=(0,0,0)),
                SwinTransformerBlock3D(dim=embed_dim, num_heads=num_heads[i_layer], window_size=(2,4,4), shift_size=(1,2,2))
            )
            self.layers.append(layer)
        self.norm = nn.LayerNorm(embed_dim)
        self.up_conv = nn.ConvTranspose3d(embed_dim, in_chans, kernel_size=(1, patch_size, patch_size), stride=(1, patch_size, patch_size))
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        x = self.patch_embed(x).permute(0, 2, 3, 4, 1) 
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x).permute(0, 4, 1, 2, 3) 
        x = self.up_conv(x)
        return self.sigmoid(x)

# ==========================================
# 3. MODEL B: CLASSIFIER
# ==========================================
class FFSwinClassifier(nn.Module):
    def __init__(self, num_classes=3, in_chans=1, embed_dim=96, patch_size=4):
        super().__init__()
        self.patch_embed = nn.Conv3d(in_chans, embed_dim, kernel_size=(1, patch_size, patch_size), stride=(1, patch_size, patch_size))
        self.layers = nn.Sequential(
            SwinTransformerBlock3D(embed_dim, num_heads=3, shift_size=(0,0,0)), 
            SwinTransformerBlock3D(embed_dim, num_heads=3, shift_size=(1,2,2)), 
            SwinTransformerBlock3D(embed_dim, num_heads=3, shift_size=(0,0,0)), 
            SwinTransformerBlock3D(embed_dim, num_heads=3, shift_size=(1,2,2))  
        )
        self.norm = nn.LayerNorm(embed_dim)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Linear(embed_dim, num_classes)
    def forward(self, x):
        x = self.patch_embed(x).permute(0, 2, 3, 4, 1) 
        x = self.layers(x)
        x = self.norm(x) # Target Layer for Grad-CAM
        B, D, H, W, C = x.shape
        x = x.view(B, -1, C).transpose(1, 2) 
        x = self.avgpool(x).flatten(1)       
        return self.head(x)

# ==========================================
# 4. EXPLAINABILITY (GRAD-CAM + CONTOURS)
# ==========================================
class GradCAM3D_FFSwin:
    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None
        target_layer.register_forward_hook(self.save_activation)
        target_layer.register_full_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output):
        self.activations = output

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def __call__(self, x, class_idx=None):
        self.model.zero_grad()
        output = self.model(x)
        if class_idx is None: class_idx = torch.argmax(output)
        score = output[0, class_idx]
        score.backward()
        
        gradients = self.gradients
        activations = self.activations
        
        # Average gradients over Depth, Height, Width
        pooled_gradients = torch.mean(gradients, dim=[0, 1, 2, 3])
        
        # Weight channels
        for i in range(activations.shape[-1]):
            activations[:, :, :, :, i] *= pooled_gradients[i]
            
        # Generate Heatmap
        heatmap = torch.mean(activations, dim=-1).squeeze()
        heatmap = F.relu(heatmap)
        if torch.max(heatmap) > 0:
            heatmap /= torch.max(heatmap)
            
        return heatmap.cpu().detach().numpy(), class_idx.item()

# ==========================================
# 5. EXECUTION PIPELINE
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# --- PATHS ---
path_denoiser = r"models\denoiser.pth"
path_classifier = r"FFSwin_Upper_Bound_Capacity\models\classifier_best.pth" 
if os.path.exists(r"FFSwin_Upper_Bound_Capacity\models\classifier_best.pth"):
    path_classifier = "FFSwin_Upper_Bound_Capacity\models\classifier_best.pth"


# Load Models
denoiser = OCT3DDenoisingAutoencoder().to(device)
if os.path.exists(path_denoiser):
    denoiser.load_state_dict(torch.load(path_denoiser, map_location=device))
    denoiser.eval()
    print("✅ Denoiser Ready.")
else:
    print(f"⚠️ Warning: Denoiser not found at {path_denoiser}")
    denoiser = None

classifier = FFSwinClassifier(num_classes=3).to(device)
if os.path.exists(path_classifier):
    classifier.load_state_dict(torch.load(path_classifier, map_location=device))
    classifier.eval()
    print("✅ Classifier Ready.")
else:
    print(f"❌ Error: Classifier not found.")

# Hook Grad-CAM
grad_cam = GradCAM3D_FFSwin(classifier, classifier.norm)

def preprocess_raw_patient(folder_path, img_size=256):
    files = sorted(glob.glob(os.path.join(folder_path, "*.bmp")))
    if not files: return None
    vol = []
    for f in files:
        img = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            img = cv2.resize(img, (img_size, img_size))
            vol.append(img)
    if not vol: return None
    if len(vol) % 2 != 0: vol.append(vol[-1])
    tensor = torch.FloatTensor(np.array(vol) / 255.0).unsqueeze(0).unsqueeze(0)
    return tensor

# --- UI ---
path_input = widgets.Text(placeholder='Paste folder path...', description='Path:', layout=widgets.Layout(width='60%'))
is_clean_checkbox = widgets.Checkbox(value=False, description='Already Denoised?', indent=False)
run_btn = widgets.Button(description='Diagnose & Mark', button_style='success', icon='crosshairs')
out = widgets.Output()

def process_patient(b):
    with out:
        clear_output()
        folder = path_input.value.strip().replace('"', '').replace("'", "")
        if not os.path.exists(folder):
            print("❌ Invalid Path")
            return
            
        # 1. Load Data
        raw_tensor = preprocess_raw_patient(folder).to(device)
        
        # 2. Smart Denoise
        if is_clean_checkbox.value:
            print("⏩ Skipping Denoising...")
            tensor_for_model = raw_tensor
        else:
            if denoiser:
                print("🌊 Running Denoising Autoencoder...")
                with torch.no_grad():
                    tensor_for_model = denoiser(raw_tensor)
            else:
                tensor_for_model = raw_tensor
        
        # 3. Classify & Explain
        print("🧠 Diagnosing...")
        
        # Grad-CAM (Heatmap)
        with torch.enable_grad():
             heatmap_3d, pred_idx = grad_cam(tensor_for_model)
        
        # Probs
        with torch.no_grad():
            logits = classifier(tensor_for_model)
            probs = F.softmax(logits, dim=1)
            confidence = probs[0][pred_idx]

        classes = ['Dry AMD', 'Wet AMD', 'NonAMD']
        diagnosis = classes[pred_idx]
        
        # 4. MARKING LOGIC (Contours & Boxes)
        mid_idx = raw_tensor.shape[2] // 2
        
        # Images
        raw_slice = raw_tensor.cpu().numpy()[0, 0, mid_idx, :, :]
        input_slice = tensor_for_model.cpu().detach().numpy()[0, 0, mid_idx, :, :]
        heatmap_slice = heatmap_3d[mid_idx]
        
        # Resize Heatmap
        heatmap_resized = cv2.resize(heatmap_slice, (input_slice.shape[1], input_slice.shape[0]))
        
        # --- CREATE MARKED IMAGE ---
        # 1. Threshold to get binary mask
        _, binary_mask = cv2.threshold(heatmap_resized, 0.4, 1, cv2.THRESH_BINARY)
        binary_mask_uint8 = np.uint8(255 * binary_mask)
        
        # 2. Find Contours
        contours, _ = cv2.findContours(binary_mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        # 3. Draw on Image
        # Convert grayscale to RGB for color drawing
        marked_img = cv2.cvtColor(np.uint8(255 * input_slice), cv2.COLOR_GRAY2RGB)
        
        # Draw Contours (Green)
        cv2.drawContours(marked_img, contours, -1, (0, 255, 0), 2)
        
        # Draw Bounding Box (Red)
        for c in contours:
            x, y, w, h = cv2.boundingRect(c)
            if w > 10 and h > 10: # Filter small noise
                cv2.rectangle(marked_img, (x, y), (x + w, y + h), (255, 0, 0), 2)
        
        # 5. Plot 3 Panels
        fig, ax = plt.subplots(1, 3, figsize=(18, 6))
        
        ax[0].imshow(raw_slice, cmap='gray')
        ax[0].set_title("1. Raw Input")
        ax[0].axis('off')
        
        ax[1].imshow(input_slice, cmap='gray')
        title_mid = "2. Denoised Input" if not is_clean_checkbox.value else "2. Clean Input"
        ax[1].set_title(title_mid)
        ax[1].axis('off')
        
        # PANEL 3: MARKED
        ax[2].imshow(marked_img)
        ax[2].set_title(f"3. Pathology Delineation\n(RED Marked ROI)", color='red', fontweight='bold')
        ax[2].axis('off')
        
        plt.suptitle(f"Diagnosis: {diagnosis} | Confidence: {confidence.item()*100:.1f}%", color='red', fontweight='bold',fontsize=18)
        plt.tight_layout()
        plt.show()
        
        print(f"📂 Patient: {os.path.basename(folder)}")
        print(f"✅ Success: The Red Box indicates the Region of Interest (ROI) for the doctor.")

run_btn.on_click(process_patient)
display(widgets.VBox([widgets.HBox([path_input, is_clean_checkbox]), run_btn]))
display(out)

<>:200: SyntaxWarning: invalid escape sequence '\M'
<>:200: SyntaxWarning: invalid escape sequence '\M'
C:\Users\Chinmay\AppData\Local\Temp\ipykernel_40348\1770465620.py:200: SyntaxWarning: invalid escape sequence '\M'
  path_classifier = "G:\My Drive\OCT_Project_PhD_Implementation\FFSwin_Pseudo_3D_Model_main\models\classifier_best_100_used_here_as_final.pth"


✅ Denoiser Ready.
✅ Classifier Ready.


C:\Users\Chinmay\AppData\Local\Temp\ipykernel_40348\1770465620.py:206: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  denoiser.load_state_dict(torch.load(path_denoiser, map_l

Output()